# RPOE-X — GRPO Training on Google Colab

**OpenEnv Hackathon Submission** | [HF Space](https://bharavi-rpoe-x.hf.space) | [GitHub](https://huggingface.co/spaces/Bharavi/rpoe_x)

Train a Rotary Parking SRE agent using **GRPO** with HF TRL. The agent learns to route
cars across a 5-zone, 20-wheel rotary parking system in HITEC City, Hyderabad.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://bharavi-rpoe-x.hf.space) (RPOE-X server) |
| Training | This Colab notebook (T4 / A100 GPU) |
| Model | `Qwen/Qwen2.5-0.5B-Instruct` + LoRA via PEFT |
| Framework | HF TRL v0.29 GRPO with vLLM backend |

### Prerequisites
- Google Colab **A100** runtime (40 GB) for vLLM colocate. T4 works with `use_vllm=False`.
- HuggingFace account with write access token saved as `HF_TOKEN` in Colab Secrets.
- The HF Space `bharavi-rpoe-x.hf.space` must be **running** (not sleeping).

> Run cells **top to bottom**. Health check in step 3 verifies the Space before loading the model.

## 1. Install Dependencies

In [ ]:
!pip install -q \
    "trl[vllm]==0.29.0" \
    "vllm>=0.11.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "pandas" \
    "numpy" \
    "requests" \
    "nest_asyncio"

# Install rpoe_x package (environment client + training utilities from HF Space)
!pip install -q "git+https://huggingface.co/spaces/Bharavi/rpoe_x"

import os
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
print("Installation complete")

In [ ]:
import warnings, logging, os

# Suppress Python warnings
warnings.filterwarnings("ignore")

# Suppress verbose library loggers
for _logger in ("transformers", "trl", "peft", "accelerate", "vllm",
                "datasets", "huggingface_hub", "torch"):
    logging.getLogger(_logger).setLevel(logging.ERROR)

# Suppress HuggingFace environment warnings
os.environ["TOKENIZERS_PARALLELISM"]  = "false"   # avoids tokenizer fork warning
os.environ["TRANSFORMERS_VERBOSITY"]  = "error"
os.environ["ACCELERATE_LOG_LEVEL"]    = "error"

print("Warnings suppressed.")

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets
(key icon in the left sidebar).

In [ ]:
import os

# HuggingFace token
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not set. Add it via Colab Secrets (left sidebar -> key icon).")

# Environment server (HF Space)
ENV_URL = "https://bharavi-rpoe-x.hf.space"

# Model
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
HUB_REPO  = "Bharavi/rpoe-x-qwen-0.5b-grpo"

# Training hyperparameters
NUM_EPISODES    = 60    # dataset size — each triggers one live rollout
MAX_STEPS       = 300   # GRPO gradient update steps
NUM_GENERATIONS = 4     # completions per prompt for group-relative comparison
MAX_TURNS       = 50    # env steps per training episode
EPISODE_MAX_STEPS = MAX_TURNS  # alias used by reward normalisation

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Max steps   : {MAX_STEPS}")
print(f"Max turns   : {MAX_TURNS}")

## 3. Health Check — Verify the Space is Running

Hit `/health` before spending time loading the model. Retries automatically if the Space is
sleeping (HF Spaces wake on first request but take ~15s).

In [ ]:
import requests, time

def wake_and_verify(base_url: str, retries: int = 5, wait: int = 15) -> bool:
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{base_url}/health", timeout=30)
            if r.status_code == 200:
                print(f"Space is alive: {r.json()}")
                return True
        except Exception:
            pass
        print(f"  Attempt {attempt}/{retries} — space sleeping, retrying in {wait}s...")
        time.sleep(wait)
    return False

alive = wake_and_verify(ENV_URL)
if not alive:
    raise RuntimeError(
        f"Cannot reach {ENV_URL}/health after several retries.\n"
        "Visit https://huggingface.co/spaces/Bharavi/rpoe_x to wake the Space, "
        "then re-run this cell."
    )

## 4. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment, and step once to confirm round-trip works.
Also defines the sync HTTP helpers (`RPOEXHTTPClient`, `_obs_to_text`, `_greedy_zone_id`)
that are used in evaluation and as fallbacks throughout the notebook.

In [ ]:
import asyncio, requests, nest_asyncio
nest_asyncio.apply()

# ── Import HTTP client + observation helpers from package ──────────────────
try:
    from rpoe_x.training.train import (
        ZONE_WHEELS,
        RPOEXHTTPClient,
        _obs_to_text,
        _zone_obs_to_text,
        _greedy_zone_id,
        _greedy_wheel_id,
    )
    print("Imported env helpers from rpoe_x.training.train")

except ImportError as e:
    print(f"Package import failed ({e}) — using inline fallbacks.")

    ZONE_WHEELS = [4, 4, 5, 4, 3]

    class RPOEXHTTPClient:
        def __init__(self, base_url, task_id="task2_medium", timeout=30):
            self.base_url   = base_url.rstrip("/")
            self.task_id    = task_id
            self.timeout    = timeout
            self.session_id = None

        def reset(self, seed=None):
            payload = {"task_id": self.task_id}
            if seed is not None:
                payload["seed"] = seed
            r = requests.post(f"{self.base_url}/reset", json=payload, timeout=self.timeout)
            r.raise_for_status()
            data = r.json()
            self.session_id = data.get("session_id")
            return data

        def step(self, action, zone_id, wheel_id=None):
            payload = {"action": action, "zone_id": zone_id}
            if wheel_id is not None:
                payload["wheel_id"] = wheel_id
            if self.session_id:
                payload["session_id"] = self.session_id
            r = requests.post(f"{self.base_url}/step", json=payload, timeout=self.timeout)
            r.raise_for_status()
            return r.json()

    def _obs_to_text(obs):
        zone_occ = [round(x, 2) for x in obs.get("zone_occupancy", [])]
        zone_q   = obs.get("zone_queue_lengths", [])
        zone_w   = [round(x, 1) for x in obs.get("zone_avg_wait", [])]
        return (
            f"step={obs.get('step', 0)} time={round(obs.get('time_of_day', 0.0), 3)}\n"
            f"zone_occupancy={zone_occ}\nzone_queue_lengths={zone_q}\nzone_avg_wait={zone_w}\n"
            "Which zone_id (0-4) should the next car route to?"
        )

    def _zone_obs_to_text(obs, zone_id):
        n = ZONE_WHEELS[zone_id]
        zone_obs_list = obs.get("zone_observations", [])
        if zone_id < len(zone_obs_list):
            z         = zone_obs_list[zone_id]
            wheel_occ = [round(x, 2) for x in z.get("wheel_occupancy", [0.0] * n)]
            wheel_q   = z.get("wheel_queue_lengths", [0] * n)
            rot_cost  = [round(x, 1) for x in z.get("est_rotation_cost", [3.0] * n)]
        else:
            occ       = obs.get("zone_occupancy", [0.0] * 5)[zone_id]
            wheel_occ = [round(occ, 2)] * n
            wheel_q   = [obs.get("zone_queue_lengths", [0] * 5)[zone_id] // max(n, 1)] * n
            rot_cost  = [3.0] * n
        tod = round(obs.get("time_of_day", 0.0), 3)
        return (
            f"zone_id={zone_id} n_wheels={n} time={tod}\n"
            f"wheel_occupancy={wheel_occ}\nwheel_queue_lengths={wheel_q}\n"
            f"est_rotation_cost={rot_cost}\nWhich wheel_id (0-{n-1}) should the car be assigned to?"
        )

    def _greedy_zone_id(obs):
        queues = obs.get("zone_queue_lengths", [0] * 5)
        occs   = obs.get("zone_occupancy", [0.0] * 5)
        max_q  = max(queues) if queues else 0
        if max_q == 0:
            return 2
        candidates = [i for i, q in enumerate(queues) if q == max_q]
        return min(candidates, key=lambda i: occs[i])

    def _greedy_wheel_id(zone_id, obs):
        n = ZONE_WHEELS[zone_id]
        zone_obs_list = obs.get("zone_observations", [])
        if zone_id < len(zone_obs_list):
            z        = zone_obs_list[zone_id]
            wheel_q  = z.get("wheel_queue_lengths", [0] * n)
            rot_cost = z.get("est_rotation_cost", [3.0] * n)
            return min(range(n), key=lambda w: wheel_q[w] + rot_cost[w])
        return 0


# ── Smoke test ─────────────────────────────────────────────────────────────
_client = RPOEXHTTPClient(base_url=ENV_URL)
_data   = _client.reset()
_obs    = _data.get("observation", {})
print(f"Zone occupancy : {[round(x, 2) for x in _obs.get('zone_occupancy', [])]}")
print(f"Queue lengths  : {_obs.get('zone_queue_lengths', [])}")
_resp = _client.step("route_to_zone", 2, wheel_id=0)
print(f"Smoke step reward={_resp.get('reward', 0.0):.3f}")
print("\nSmoke test passed. Environment is ready for training.")

## 5. Import Training Utilities

Imports `ORCH_SYSTEM`, `ZONE_SYSTEM`, `reward_total`, `build_rollout_func`, and `plot_rewards`
from the `rpoe_x` package. If the package doesn't export these symbols, inline fallbacks activate
automatically so training continues uninterrupted.

- **`ORCH_SYSTEM`** — system prompt for the orchestrator (zone routing)
- **`ZONE_SYSTEM`** — system prompt for the zone agents (wheel assignment)
- **`build_rollout_func`** — drives orchestrator ↔ env interaction during Phase 1
- **`reward_total`** — passes env rewards back to GRPO for both agents

In [ ]:
import logging, re, math, torch, nest_asyncio
from datetime import datetime
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ── Import training utilities from package ─────────────────────────────────
try:
    from rpoe_x.training.train import (
        ORCH_SYSTEM,
        ZONE_SYSTEM,
        _proxy_step_reward,
        reward_total,
        build_rollout_func,
        plot_rewards,
    )
    print("Imported training utilities from rpoe_x.training.train")

except ImportError as e:
    print(f"Package import failed ({e}) — using inline fallbacks.")

    ORCH_SYSTEM = """\
You are an AI orchestrator for a multi-tower rotary parking system in HITEC City, Hyderabad.
Every turn you receive the current parking state and must respond with the best zone_id (0-4).

Zones:
  0 = Cyber Towers Junction  (4 wheels, 48 slots, high demand)
  1 = Inorbit Mall Signal    (4 wheels, 48 slots, moderate demand)
  2 = Hitech City Metro      (5 wheels, 60 slots, LARGEST BUFFER)
  3 = Mindspace Junction     (4 wheels, 48 slots, moderate demand)
  4 = Kondapur / Whitefields (3 wheels, 36 slots, low demand)

Strategy: route to the zone with the highest queue_length.
Prefer zone 2 when queues are equal. Never route to a full zone (occupancy=1.0)."""

    ZONE_SYSTEM = """\
You are a zone agent for a rotary parking tower in HITEC City, Hyderabad.
You receive the wheel-level state of your assigned zone and must pick the best wheel_id.

Strategy:
- Pick the wheel with the lowest wheel_queue_lengths (fewest filled slots).
- Break ties using est_rotation_cost (prefer wheels needing fewer rotation steps).
- Never pick a fully occupied wheel (wheel_occupancy = 1.0) if alternatives exist."""

    def _proxy_step_reward(pre_obs, post_obs, zone_id):
        pre_q = pre_obs.get("zone_queue_lengths", [0] * 5)
        routing_score = 1.0 if zone_id < len(pre_q) and pre_q[zone_id] > 0 else 0.0
        occ      = post_obs.get("zone_occupancy", [0.5] * 5)
        mean_occ = sum(occ) / len(occ) if occ else 0.5
        std_occ  = (sum((x - mean_occ) ** 2 for x in occ) / len(occ)) ** 0.5 if occ else 0.0
        balance_score = max(0.0, 1.0 - std_occ / 0.5)
        return 0.7 * routing_score + 0.3 * balance_score

    def reward_total(completions, **kwargs):
        return [float(c.get("reward", 0.0)) if isinstance(c, dict) else 0.0 for c in completions]

    def plot_rewards(trainer, save_path=None):
        import pandas as pd, matplotlib.pyplot as plt
        log_df = pd.DataFrame(trainer.state.log_history)
        reward_cols = [c for c in log_df.columns if "reward" in c.lower()]
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for col in reward_cols:
            if "step" in log_df.columns:
                axes[0].plot(log_df["step"], log_df[col], label=col)
        axes[0].set_title("GRPO Reward"); axes[0].set_xlabel("Step"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
        if "loss" in log_df.columns and "step" in log_df.columns:
            axes[1].plot(log_df["step"], log_df["loss"], color="orange")
            axes[1].set_title("Training Loss"); axes[1].set_xlabel("Step"); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=120)
        plt.show()

    def build_rollout_func(env_url, tokenizer, max_turns=50):
        _device = "cuda" if torch.cuda.is_available() else "cpu"

        def _infer(model, tok, messages, lo, hi):
            prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inp = tok(prompt, return_tensors="pt").to(_device)
            was_training = model.training
            model.eval()
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=16, temperature=0.9,
                                     do_sample=True, pad_token_id=tok.eos_token_id)
            if was_training:
                model.train()
            text = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
            for d in re.findall(r"\b(\d+)\b", text):
                if lo <= int(d) <= hi:
                    return int(d)
            return lo

        def _rollout(prompts, model=None, processing_class=None, **kwargs):
            tok = processing_class or tokenizer
            all_completions, all_rewards = [], []
            for _ in prompts:
                client    = RPOEXHTTPClient(base_url=env_url, task_id="task2_medium")
                data      = client.reset()
                obs       = data.get("observation", {})
                ep_reward = 0.0
                turns     = 0
                text      = ""
                while not obs.get("done", False) and turns < max_turns:
                    pre_obs  = obs
                    zone_id  = _infer(model, tok, [
                        {"role": "system", "content": ORCH_SYSTEM},
                        {"role": "user",   "content": _obs_to_text(obs)},
                    ], lo=0, hi=4)
                    n_wheels = ZONE_WHEELS[zone_id]
                    wheel_id = _infer(model, tok, [
                        {"role": "system", "content": ZONE_SYSTEM},
                        {"role": "user",   "content": _zone_obs_to_text(obs, zone_id)},
                    ], lo=0, hi=n_wheels - 1)
                    resp      = client.step("route_to_zone", zone_id, wheel_id=wheel_id)
                    obs       = resp.get("observation", {})
                    ep_reward += _proxy_step_reward(pre_obs, obs, zone_id)
                    text      += f"{zone_id},{wheel_id};"
                    turns     += 1
                normalised = min(1.0, ep_reward / max(max_turns, 1))
                all_completions.append({"reward": normalised, "content": text})
                all_rewards.append(normalised)
            return all_completions, all_rewards

        return _rollout


print("\nTraining utilities ready.")
print(f"ORCH_SYSTEM : {ORCH_SYSTEM[:55]}...")
print(f"ZONE_SYSTEM : {ZONE_SYSTEM[:55]}...")

## 6. Training Setup

One model, one LoRA adapter, one training loop — the same Qwen2.5-0.5B handles both roles.
The system prompt injected by the rollout function determines the role each call:
`ORCH_SYSTEM` for zone routing, `ZONE_SYSTEM` for wheel assignment.

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA config — applied by GRPOTrainer on init
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Rollout function — one live episode per dataset entry
rollout_func = build_rollout_func(ENV_URL, tokenizer, max_turns=MAX_TURNS)

# Dataset: simple trigger prompts; full prompt is built inside rollout_func
dataset = Dataset.from_dict({"prompt": ["Route the next car."] * NUM_EPISODES})

# Timestamped output directory
timestamp  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f"outputs/rpoe-x-grpo-{timestamp}"
Path(output_dir).mkdir(parents=True, exist_ok=True)

print(f"Output : {output_dir}")
print(f"Dataset: {len(dataset)} episodes (one live rollout each)")
print(f"LoRA   : r=16, alpha=16, target_modules=q/k/v/o/gate/up/down")

In [ ]:
import sys, io, subprocess

# ── GPU detection ─────────────────────────────────────────────────────────
def _detect_gpu() -> tuple[str, bool]:
    """Returns (gpu_name, is_a100)."""
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL, text=True
        ).strip().splitlines()[0]
        return out, "A100" in out.upper()
    except Exception:
        return "unknown", False

GPU_NAME, IS_A100 = _detect_gpu()
USE_VLLM = IS_A100   # vLLM colocate requires A100; T4/V100 use HF generate
print(f"GPU detected : {GPU_NAME}")
print(f"vLLM backend : {'ENABLED (colocate)' if USE_VLLM else 'DISABLED — using HF generate (T4 mode)'}")
if not IS_A100:
    print("  TIP: Switch to an A100 runtime (Runtime → Change runtime type) for 4× faster training.")

# ── Colab fileno patch (vLLM colocate only) ───────────────────────────────
if USE_VLLM:
    def _patch_fileno(stream, fallback_fd: int) -> None:
        original = stream.fileno
        def _fileno():
            try:
                return original()
            except io.UnsupportedOperation:
                return fallback_fd
        stream.fileno = _fileno
    _patch_fileno(sys.stdout, 1)
    _patch_fileno(sys.stderr, 2)

# ── GRPO Config ───────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    use_vllm                      = USE_VLLM,
    vllm_mode                     = "colocate" if USE_VLLM else None,
    output_dir                    = output_dir,
    learning_rate                 = 5e-6,
    lr_scheduler_type             = "cosine",
    warmup_steps                  = 20,
    max_steps                     = MAX_STEPS,
    per_device_train_batch_size   = 2 if IS_A100 else 1,   # T4 has less VRAM
    gradient_accumulation_steps   = 4,
    generation_batch_size         = NUM_GENERATIONS,
    num_generations               = NUM_GENERATIONS,
    max_completion_length         = 512,
    temperature                   = 0.9,
    logging_steps                 = 10,
    save_strategy                 = "no",
    report_to                     = "none",
    seed                          = 42,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    beta                          = 0.001,
)

trainer = GRPOTrainer(
    model            = MODEL_ID,
    processing_class = tokenizer,
    reward_funcs     = [reward_total],
    args             = grpo_config,
    train_dataset    = dataset,
    rollout_func     = rollout_func,
    peft_config      = peft_config,
)

print("\nGRPOTrainer initialized")
print(f"  Backend        : {'vLLM colocate (A100)' if USE_VLLM else 'HF generate (T4)'}")
print(f"  Batch size     : {grpo_config.per_device_train_batch_size} × {grpo_config.gradient_accumulation_steps} accum × {NUM_GENERATIONS} gen")
print(f"  Reward source  : env.step() via rollout_func (orchestrator + zone agent)")

## 7. Train

Each step: sample prompt → generate `NUM_GENERATIONS` completions via `rollout_func`
(2 model calls per env step: orchestrator role then zone agent role) → `reward_total` → GRPO update.

In [ ]:
print("Starting GRPO training ...")
print(f"  Model       : {MODEL_ID}")
print(f"  Max steps   : {MAX_STEPS}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Dataset     : {len(dataset):,} examples")
print()

trainer.train()

# Save model + tokenizer (tokenizer must be saved separately for eval reload)
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\nModel + tokenizer saved to {output_dir}")

## 8. Reward Curves

Visualise training progress. Uses `plot_rewards` from the package when available,
falling back to the inline matplotlib version defined in step 5.

In [ ]:
try:
    plot_rewards(trainer, save_path=f"{output_dir}/reward_curve.png")
except Exception as e:
    print(f"plot_rewards failed ({e}), using raw log history...")
    import pandas as pd, matplotlib.pyplot as plt
    log_df = pd.DataFrame(trainer.state.log_history)
    reward_cols = [c for c in log_df.columns if "reward" in c.lower()]
    fig, ax = plt.subplots(figsize=(10, 4))
    for col in reward_cols:
        if "step" in log_df.columns:
            ax.plot(log_df["step"], log_df[col], label=col)
    ax.set_title("GRPO Reward During Training")
    ax.set_xlabel("Step"); ax.set_ylabel("Reward")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/reward_curve.png", dpi=120)
    plt.show()
    print(f"Saved to {output_dir}/reward_curve.png")

## 9. Evaluate — Single Trained Model (Both Roles)

Load the single trained adapter and use it for both orchestrator and zone agent decisions.
Compare against the greedy baseline (greedy zone + greedy wheel). Saves scores to `trained_model_scores.json`.

> If CUDA OOM occurs here, restart the runtime to clear GPU state from training, then re-run this cell only.

In [ ]:
import torch, json, re, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# trainer.save_model() saves only the LoRA adapter weights — NOT the full model.
# Must load base model first, then attach the adapter via PeftModel.
eval_tok  = AutoTokenizer.from_pretrained(output_dir)
eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
eval_model = PeftModel.from_pretrained(eval_base, output_dir)
eval_model.eval()
print(f"Trained adapter loaded: base={MODEL_ID}, adapter={output_dir}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def _infer_eval(messages, lo, hi):
    """Single forward pass, returns int in [lo, hi]."""
    prompt = eval_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = eval_tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = eval_model.generate(
            **inp, max_new_tokens=16, temperature=0.1,
            do_sample=True, pad_token_id=eval_tok.eos_token_id,
        )
    text = eval_tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    for d in re.findall(r'\b(\d+)\b', text):
        if lo <= int(d) <= hi:
            return int(d)
    return lo


def run_trained_episode(task_id: str, max_steps: int) -> dict:
    """Full multi-agent episode: single model acts as orchestrator then zone agent each step."""
    client = RPOEXHTTPClient(base_url=ENV_URL, task_id=task_id)
    data   = client.reset(seed=42)
    obs    = data.get("observation", {})
    total_reward, steps = 0.0, 0
    t0 = time.time()
    while not obs.get("done", False) and steps < max_steps:
        zone_id  = _infer_eval(
            [{"role": "system", "content": ORCH_SYSTEM},
             {"role": "user",   "content": _obs_to_text(obs)}], lo=0, hi=4)
        wheel_id = _infer_eval(
            [{"role": "system", "content": ZONE_SYSTEM},
             {"role": "user",   "content": _zone_obs_to_text(obs, zone_id)}],
            lo=0, hi=ZONE_WHEELS[zone_id] - 1)
        resp  = client.step("route_to_zone", zone_id, wheel_id=wheel_id)
        obs   = resp.get("observation", {})
        total_reward += float(resp.get("reward", 0.0))
        steps += 1
    score = obs.get("score", total_reward / max(steps, 1))
    return {"task_id": task_id, "score": round(float(score), 4),
            "steps": steps, "elapsed": round(time.time() - t0, 1)}


def run_greedy_episode(task_id: str, max_steps: int) -> dict:
    """Baseline: greedy zone selection + greedy wheel selection."""
    client = RPOEXHTTPClient(base_url=ENV_URL, task_id=task_id)
    data   = client.reset(seed=42)
    obs    = data.get("observation", {})
    total_reward, steps = 0.0, 0
    t0 = time.time()
    while not obs.get("done", False) and steps < max_steps:
        zone_id  = _greedy_zone_id(obs)
        wheel_id = _greedy_wheel_id(zone_id, obs)
        resp     = client.step("route_to_zone", zone_id, wheel_id=wheel_id)
        obs      = resp.get("observation", {})
        total_reward += float(resp.get("reward", 0.0))
        steps += 1
    score = obs.get("score", total_reward / max(steps, 1))
    return {"task_id": task_id, "score": round(float(score), 4),
            "elapsed": round(time.time() - t0, 1)}


TASK_STEPS = {"task1_easy": 200, "task2_medium": 400, "task3_hard": 1080}
trained_results, greedy_results = {}, {}

print("\n--- Evaluation: Trained (orch + zone, single model) vs Greedy baseline ---")
for tid, steps in TASK_STEPS.items():
    print(f"\n{tid} ({steps} steps):")
    try:
        r = run_trained_episode(tid, steps)
        trained_results[tid] = r
        print(f"  Trained : score={r['score']:.4f}  ({r['elapsed']}s)")
    except Exception as e:
        print(f"  Trained : FAILED — {e}")
        trained_results[tid] = {"task_id": tid, "score": 0.0, "error": str(e)}
    try:
        g = run_greedy_episode(tid, steps)
        greedy_results[tid] = g
        print(f"  Greedy  : score={g['score']:.4f}  ({g['elapsed']}s)")
    except Exception as e:
        print(f"  Greedy  : FAILED — {e}")
        greedy_results[tid] = {"task_id": tid, "score": 0.0}

trained_avg = sum(r["score"] for r in trained_results.values()) / len(trained_results)
greedy_avg  = sum(r["score"] for r in greedy_results.values()) / len(greedy_results)
print(f"\nAverage — Trained: {trained_avg:.4f}  Greedy: {greedy_avg:.4f}")
print(f"Improvement: +{trained_avg - greedy_avg:.4f}")

with open("trained_model_scores.json", "w") as f:
    json.dump({
        "trained": {k: v["score"] for k, v in trained_results.items()},
        "greedy":  {k: v["score"] for k, v in greedy_results.items()},
        "trained_avg": round(trained_avg, 4),
        "greedy_avg":  round(greedy_avg, 4),
    }, f, indent=2)
print("Scores saved to trained_model_scores.json")

## 10. Push to Hub (Optional)

Upload the trained LoRA adapter to HuggingFace Hub.

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
api.create_repo(repo_id=HUB_REPO, repo_type="model", exist_ok=True)

# Single adapter covers both orchestrator and zone agent roles
api.upload_folder(
    folder_path    = output_dir,
    repo_id        = HUB_REPO,
    repo_type      = "model",
    commit_message = f"GRPO adapter (orch + zone agent, shared model) — {timestamp}",
)
print(f"Pushed to https://huggingface.co/{HUB_REPO}")

---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `Cannot reach /health` | Visit the HF Space URL in browser to wake it, then re-run step 3. |
| `ImportError: rpoe_x.training.train` | Inline fallbacks activate automatically — check the console for confirmation. |
| `vllm_mode colocate` crash / OOM | Must be on A100. On T4: set `use_vllm=False` and remove `vllm_mode`. |
| CUDA OOM during training | Reduce `NUM_GENERATIONS` to 2 or `per_device_train_batch_size` to 1. |
| `max_concurrent_envs` error | Add `max_concurrent_envs=64` to `create_app()` in `server/app.py`. |
| Reward stuck at 0 | Verify smoke test passed; confirm `ENV_URL` is reachable from Colab. |
| `rollout_func` not accepted by GRPOTrainer | Ensure `trl==0.29.0`; older versions use `environment_factory` instead. |
| Eval cell CUDA OOM | Restart runtime to clear GPU state from training, then re-run eval cell only. |
| `tokenizer has no attribute pad_token` | Already handled — cell 10 sets `pad_token = eos_token` if missing. |

## Required server change for parallel training

GRPO generates `NUM_GENERATIONS` completions in parallel. Without this the server
rejects concurrent sessions:

```python
# server/app.py
app = create_app(
    create_environment,
    OrchestratorAction,
    OrchestratorObs,
    max_concurrent_envs=64,   # ← add this
)
```